In [ ]:
#Import the libraries
import os     #auto processes every image in subfolders
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# =====================================
# Folder
# =====================================
input_folder = "Inputs"

count = 0

for root, dirs, files in os.walk(input_folder):

    for filename in sorted(files):

        image_path = os.path.join(root, filename)
    
        if os.path.isdir(image_path):
            continue
    
        if not filename.lower().endswith((".png", ".jpg", ".jpeg")):
            continue
    
        img = cv2.imread(image_path)
    
        if img is None:
            continue
    
        # =====================================
        # Image Preprocessing
        # =====================================
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
        blurred = cv2.GaussianBlur(img_rgb, (5,5), 0)
    
        hsv = cv2.cvtColor(blurred, cv2.COLOR_RGB2HSV)
    
        # =====================================
        # Temporary Colour Segmentation
        # (Used only to generate input for
        # testing the shape detection module)
        # =====================================
    
        # ---------- RED ----------
        lower_red1 = np.array([0,100,80])
        upper_red1 = np.array([10,255,255])
        
        lower_red2 = np.array([170,100,80])
        upper_red2 = np.array([180,255,255])
    
        red_mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
        red_mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    
        red_mask = red_mask1 + red_mask2
    
        # ---------- BLUE ----------
        lower_blue = np.array([95,120,60])
        upper_blue = np.array([135,255,255])
    
        blue_mask = cv2.inRange(hsv, lower_blue, upper_blue)
    
        # ---------- YELLOW ----------
        lower_yellow = np.array([15,80,80])
        upper_yellow = np.array([40,255,255])
    
        yellow_mask = cv2.inRange(hsv, lower_yellow, upper_yellow)
    
        # Combine all colours
        mask = red_mask | blue_mask | yellow_mask

        # =====================================
        # Morphological Cleaning
        # =====================================
        
        kernel = np.ones((3,3), np.uint8)
        
        # Remove small noise
        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_OPEN,
            kernel
        )
        
        # Fill small holes
        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_CLOSE,
            kernel
        )

        # =====================================
        # Find Contours
        # =====================================
    
        contours, hierarchy = cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )
    
        best_contour = None
        best_score = 0
        
        for cnt in contours:
        
            area = cv2.contourArea(cnt)
        
            if area < 1000:
                continue
        
            perimeter = cv2.arcLength(cnt, True)
        
            if perimeter == 0:
                continue
        
            hull = cv2.convexHull(cnt)
    
            hull_area = cv2.contourArea(hull)
            
            if hull_area == 0:
                continue
            
            solidity = area / hull_area
            
            circularity = 4 * np.pi * area / (perimeter * perimeter)
            
            score = area * circularity * solidity
        
            if score > best_score:
                best_score = score
                best_contour = cnt
    
        if best_contour is None:
            print(filename, "No sign detected")
            continue
    
        # =====================================
        # Improve Contour
        # =====================================
    
        # best_contour = cv2.convexHull(best_contour)
        hull = cv2.convexHull(best_contour)
        
        # =====================================
        # Feature Extraction
        # =====================================
        area = cv2.contourArea(best_contour)
        
        perimeter = cv2.arcLength(best_contour, True)
        
        circularity = 4*np.pi*area/(perimeter*perimeter)
        
        contour_used = best_contour
        # if circularity < 0.75:
        #     contour_used = best_contour
        # else:
        #     contour_used = cv2.convexHull(best_contour)
        
        # =====================================
        # Feature Extraction (Using Convex Hull)
        # =====================================
        
        area = cv2.contourArea(contour_used)
        
        perimeter = cv2.arcLength(contour_used, True)
        
        if perimeter == 0:
            continue
        
        circularity = 4 * np.pi * area / (perimeter * perimeter)
        
        epsilon = 0.02 * perimeter
        
        approx = cv2.approxPolyDP(
            contour_used,
            epsilon,
            True
        )
        
        vertices = len(approx)
        
        x, y, w, h = cv2.boundingRect(contour_used)
        
        aspect_ratio = w / float(h)
            
        # =====================================
        # Shape Classification
        # =====================================
            
        if vertices == 3:
            shape = "Triangle"
    
        elif vertices == 4:
        
            if 0.9 <= aspect_ratio <= 1.1:
                shape = "Square"
            else:
                shape = "Rectangle"
        
        elif circularity >= 0.82:
            shape = "Circle"
            
        elif 7 <= vertices <= 9:
            shape = "Octagon"
        
        
        
        else:
            shape = "Unknown"
    
        # =====================================
        # Draw Result
        # =====================================
    
        result = img_rgb.copy()
    
        cv2.drawContours(
            result,
            [best_contour],
            -1,
            (0,255,0),
            2
        )
    
        cv2.putText(
            result,
            shape,
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255,0,0),
            2
        )
    
        # =====================================
        # Print
        # =====================================
    
        print("=" * 40)
        print(filename)
        
        print("\nExtracted Features")
        print(f"Area         : {area:.1f}")
        print(f"Perimeter    : {perimeter:.1f}")
        print(f"Circularity  : {circularity:.2f}")
        print(f"Vertices     : {vertices}")
        print(f"Aspect Ratio : {aspect_ratio:.2f}")
        
        print("\nClassification Result")
        print(f"Detected Shape : {shape}")
        
        # =====================================
        # Display
        # =====================================
    
        plt.figure(figsize=(18,4))
    
        plt.subplot(151)
        plt.imshow(img_rgb)
        plt.title("Original")
        plt.axis("off")
    
        plt.subplot(152)
        plt.imshow(hsv)
        plt.title("HSV")
        plt.axis("off")
    
        plt.subplot(153)
        plt.imshow(mask, cmap="gray")
        plt.title("Colour Mask")
        plt.axis("off")
    
        plt.subplot(154)
        plt.imshow(result)
        plt.title("Detected Shape")
        plt.axis("off")
    
        plt.subplot(155)
        plt.imshow(cv2.drawContours(img_rgb.copy(),
                                    [best_contour],
                                    -1,
                                    (0,255,0),
                                    2))
        plt.title(shape)
        plt.axis("off")
    
        plt.tight_layout()
        plt.show()
    
        # count += 1
    
        # if count == 6:
        #     break
